# Marathos EDA from bronze layer to silver layer cleaning
- This notebook tests cleaning rules for the raw marathon Bronze table before the final logic is added to the Silver pipeline script.
- The goal is to create a cleaned marathon OBT table for the Silver layer.

- year_of_event
- event_dates
- event_name
- event_distance_length 
    - * athlete_performance (in this ntbk)
- event_number_of_finishers

In [0]:
# Load Bronze marathon table for Silver cleaning trial

marathon_bronze_df = spark.sql("""
    SELECT *
    FROM marathos_catalog.bronze.raw_marathon_results
""")

display(marathon_bronze_df)

In [0]:
print(f"Bronze marathon rows: {marathon_bronze_df.count()}")

In [0]:
marathon_bronze_df.columns

In [0]:
# imports
import re
from pyspark.sql.functions import col, when, count, to_date, try_to_date, expr, regexp_extract, concat, lit, trim, length, lower


In [0]:
# Convert column names to snake_case
def to_snake_case(column_name):
    clean_name = column_name.strip().lower()
    clean_name = re.sub(r"[^a-z0-9]+", "_", clean_name)
    return clean_name.strip("_")


# Rename all dataframe columns
def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

In [0]:
# Create Silver trial dataframe with clean column names
marathon_silver_df = rename_columns_to_snake_case(marathon_bronze_df)

display(marathon_silver_df)

In [0]:
# Check new column names
marathon_silver_df.columns

## year_of_event
- Check for NULLS again and cast to integer type

In [0]:
# Check nulls and year range
display(
    marathon_silver_df
    .select("year_of_event")
    .summary()
)

In [0]:
# Check if any event years are outside the expected dataset range
invalid_year_of_event_df = (
    marathon_silver_df
    .filter(
        (col("year_of_event").isNull()) |
        (col("year_of_event") < 1798) |
        (col("year_of_event") > 2022)
    )
)

print(f"Invalid year_of_event rows: {invalid_year_of_event_df.count()}")

In [0]:
# Cast year_of_event to integer
marathon_silver_df = marathon_silver_df.withColumn(
    "year_of_event",
    col("year_of_event").cast("int")
)

display(marathon_silver_df.select("year_of_event").summary())

#### Findings and Plan:
- The year_of_event column has no missing values and the values are within the expected historical range of the dataset, from 1798 to 2022.

- For the Silver layer, I cast year_of_event to integer to make the column consistent for filtering, aggregation, and later date/dimensional modelling.

## event_dates
- I have in my EDA different formats. Not good.
```
24.06.2017
12.-13.10.2019
31.12.-01.01.2018
28.12.2019-03.01.2020
```

- Need to create new columns with these values:
```
event_date_raw
event_date_format_type
event_start_date
event_end_date
```



In [0]:
# Keep the original event_dates
marathon_silver_df = marathon_silver_df.withColumn(
    "event_date_raw",
    col("event_dates")
)

# Classify the different date formats found in EDA
event_dates_format_df = (
    marathon_silver_df
    .withColumn(
        "event_date_format_type",
        when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"), "single_date")
        .when(col("event_dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_same_month")
        .when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_different_month")
        .when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}-\d{2}\.\d{2}\.\d{4}$"), "date_range_different_year")
        .otherwise("unknown_format")
    )
    .groupBy("event_date_format_type")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

# confirms the format groups before parsing
display(event_dates_format_df)

In [0]:
# save the date format classification
# Keep original event date and classify the date format
marathon_silver_df = (
    marathon_silver_df
    .withColumn("event_date_raw", col("event_dates"))
    .withColumn(
        "event_date_format_type",
        when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"), "single_date")
        .when(col("event_dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_same_month")
        .when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_different_month")
        .when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}-\d{2}\.\d{2}\.\d{4}$"), "date_range_different_year")
        .otherwise("unknown_format")
    )
)

display(
    marathon_silver_df.select(
        "event_dates",
        "event_date_raw",
        "event_date_format_type"
    )
)


In [0]:
# Parse simple single-date events into a proper date column
# Use try_to_date to handle invalid dates gracefully (returns NULL instead of error)
marathon_silver_df = marathon_silver_df.withColumn(
    "event_start_date",
    when(
        col("event_date_format_type") == "single_date",
        try_to_date(col("event_dates"), "dd.MM.yyyy")
    )
)

display(
    marathon_silver_df
    .filter(col("event_date_format_type") == "single_date")
    .select("event_dates", "event_date_format_type", "event_start_date")
)

#### look for malformed dates 00.06.1991

In [0]:
# Parse single-date events safely.
# Invalid dates, like 00.06.1991, become NULL instead of breaking the notebook.
marathon_silver_df = marathon_silver_df.withColumn(
    "event_start_date",
    when(
        col("event_date_format_type") == "single_date",
        expr("try_to_date(event_dates, 'dd.MM.yyyy')")
    )
)

display(
    marathon_silver_df
    .filter(col("event_date_format_type") == "single_date")
    .select("event_dates", "event_date_format_type", "event_start_date")
)

In [0]:
# Count single-date rows that could not be parsed
failed_single_date_parse_count = (
    marathon_silver_df
    .filter(
        (col("event_date_format_type") == "single_date") &
        (col("event_start_date").isNull())
    )
    .count()
)

print(f"Failed single_date parse rows: {failed_single_date_parse_count}")

In [0]:
# Inspect invalid single-date values
invalid_single_dates_df = (
    marathon_silver_df
    .filter(
        (col("event_date_format_type") == "single_date") &
        (col("event_start_date").isNull())
    )
    .select(
        "year_of_event",
        "event_dates",
        "event_name",
        "event_distance_length"
    )
    .distinct()
    .orderBy("event_dates")
)

display(invalid_single_dates_df)

- Some event_dates values match the expected single-date pattern but are not valid real dates. 
For example, 00.06.1991 has an invalid day value.

- To prevent the Silver transformation from failing, I used try_to_date instead of to_date. 
- Invalid dates are converted to NULL and will be removed later because the event date is required for downstream analysis and date modelling.

#### parse date_range_same_month
- from `03.-09.11.1991` to
```
event_start_date = 1991-11-03
event_end_date   = 1991-11-09
```


In [0]:
# Extract parts from same-month date ranges like 03.-09.11.1991
same_month_start_day = regexp_extract(col("event_dates"), r"^(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 1)
same_month_end_day = regexp_extract(col("event_dates"), r"^(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 2)
same_month_month = regexp_extract(col("event_dates"), r"^(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 3)
same_month_year = regexp_extract(col("event_dates"), r"^(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 4)

# Create temporary date strings for parsing
marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "same_month_start_raw",
        concat(same_month_start_day, lit("."), same_month_month, lit("."), same_month_year)
    )
    .withColumn(
        "same_month_end_raw",
        concat(same_month_end_day, lit("."), same_month_month, lit("."), same_month_year)
    )
)

# Parse start date and end date safely
marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "event_start_date",
        when(
            col("event_date_format_type") == "date_range_same_month",
            expr("try_to_date(same_month_start_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_start_date"))
    )
    .withColumn(
        "event_end_date",
        when(
            col("event_date_format_type") == "single_date",
            col("event_start_date")
        ).when(
            col("event_date_format_type") == "date_range_same_month",
            expr("try_to_date(same_month_end_raw, 'dd.MM.yyyy')")
        )
    )
)

display(
    marathon_silver_df
    .filter(col("event_date_format_type") == "date_range_same_month")
    .select(
        "event_dates",
        "same_month_start_raw",
        "same_month_end_raw",
        "event_start_date",
        "event_end_date"
    )
)

In [0]:
# Check same-month date ranges that failed to parse
failed_same_month_parse_count = (
    marathon_silver_df
    .filter(
        (col("event_date_format_type") == "date_range_same_month") &
        (
            col("event_start_date").isNull() |
            col("event_end_date").isNull()
        )
    )
    .count()
)

print(f"Failed date_range_same_month parse rows: {failed_same_month_parse_count}")

#### parse date_range_different_month

In [0]:
# Extract parts from different-month ranges like 31.12.-01.01.2018
diff_month_start_day = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 1)
diff_month_start_month = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 2)
diff_month_end_day = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 3)
diff_month_end_month = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 4)
diff_month_end_year = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 5)

# Build raw date strings for start and end dates
marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "diff_month_start_raw",
        concat(
            diff_month_start_day,
            lit("."),
            diff_month_start_month,
            lit("."),
            when(
                diff_month_start_month.cast("int") > diff_month_end_month.cast("int"),
                diff_month_end_year.cast("int") - 1
            ).otherwise(diff_month_end_year)
        )
    )
    .withColumn(
        "diff_month_end_raw",
        concat(
            diff_month_end_day,
            lit("."),
            diff_month_end_month,
            lit("."),
            diff_month_end_year
        )
    )
)

# Parse different-month start and end dates safely
marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "event_start_date",
        when(
            col("event_date_format_type") == "date_range_different_month",
            expr("try_to_date(diff_month_start_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_start_date"))
    )
    .withColumn(
        "event_end_date",
        when(
            col("event_date_format_type") == "date_range_different_month",
            expr("try_to_date(diff_month_end_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_end_date"))
    )
)

display(
    marathon_silver_df
    .filter(col("event_date_format_type") == "date_range_different_month")
    .select(
        "event_dates",
        "diff_month_start_raw",
        "diff_month_end_raw",
        "event_start_date",
        "event_end_date"
    )
)

In [0]:
# Check different-month date ranges that failed to parse
failed_diff_month_parse_count = (
    marathon_silver_df
    .filter(
        (col("event_date_format_type") == "date_range_different_month") &
        (
            col("event_start_date").isNull() |
            col("event_end_date").isNull()
        )
    )
    .count()
)

print(f"Failed date_range_different_month parse rows: {failed_diff_month_parse_count}")

#### parse date_range_different_year

In [0]:
# Extract parts from full date ranges like 28.12.2019-03.01.2020
diff_year_start_day = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 1)
diff_year_start_month = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 2)
diff_year_start_year = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 3)

diff_year_end_day = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 4)
diff_year_end_month = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 5)
diff_year_end_year = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 6)

# Build raw date strings for start and end dates
marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "diff_year_start_raw",
        concat(
            diff_year_start_day,
            lit("."),
            diff_year_start_month,
            lit("."),
            diff_year_start_year
        )
    )
    .withColumn(
        "diff_year_end_raw",
        concat(
            diff_year_end_day,
            lit("."),
            diff_year_end_month,
            lit("."),
            diff_year_end_year
        )
    )
)

# Parse different-year start and end dates safely
marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "event_start_date",
        when(
            col("event_date_format_type") == "date_range_different_year",
            expr("try_to_date(diff_year_start_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_start_date"))
    )
    .withColumn(
        "event_end_date",
        when(
            col("event_date_format_type") == "date_range_different_year",
            expr("try_to_date(diff_year_end_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_end_date"))
    )
)

display(
    marathon_silver_df
    .filter(col("event_date_format_type") == "date_range_different_year")
    .select(
        "event_dates",
        "diff_year_start_raw",
        "diff_year_end_raw",
        "event_start_date",
        "event_end_date"
    )
)

In [0]:
# Check different-year date ranges that failed to parse
failed_diff_year_parse_count = (
    marathon_silver_df
    .filter(
        (col("event_date_format_type") == "date_range_different_year") &
        (
            col("event_start_date").isNull() |
            col("event_end_date").isNull()
        )
    )
    .count()
)

print(f"Failed date_range_different_year parse rows: {failed_diff_year_parse_count}")

- All event date formats are now parsed successfully except the 1002 invalid single-date rows found earlier.
- Create event_start_date and event_end_date. Remove rows where event_start_date or event_end_date is NULL.

#### remove invalid parsed dates

In [0]:
# Count rows before removing invalid parsed dates
rows_before_date_cleaning = marathon_silver_df.count()

# Remove rows where parsed event dates are missing
marathon_silver_df = marathon_silver_df.filter(
    col("event_start_date").isNotNull() &
    col("event_end_date").isNotNull()
)

# Count rows after removing invalid parsed dates
rows_after_date_cleaning = marathon_silver_df.count()

print(f"Rows before date cleaning: {rows_before_date_cleaning}")
print(f"Rows after date cleaning: {rows_after_date_cleaning}")
print(f"Rows removed: {rows_before_date_cleaning - rows_after_date_cleaning}")

In [0]:
# Confirm no invalid parsed dates remain
invalid_event_date_count = (
    marathon_silver_df
    .filter(
        col("event_start_date").isNull() |
        col("event_end_date").isNull()
    )
    .count()
)

print(f"Invalid event date rows remaining: {invalid_event_date_count}")

- The event_dates column was parsed into event_start_date and event_end_date.

- The dataset contained 1002 invalid date rows, such as dates with day value 00. These rows were removed because valid event dates are required for time-based analysis, dimensional modelling, and dashboard filtering.

- After cleaning, no invalid parsed event dates remain.

## event_name
Check again:
1. null values
2. empty strings
3. basic trimming

In [0]:
# Display invalid event_name rows, if any
invalid_event_name_df = (
    marathon_silver_df
    .filter(
        col("event_name").isNull() |
        (length(trim(col("event_name"))) == 0)
    )
    .select(
        "year_of_event",
        "event_dates",
        "event_name",
        "event_distance_length",
        "athlete_country"
    )
)

display(invalid_event_name_df)

In [0]:
# Remove extra spaces before/after event_name
marathon_silver_df = marathon_silver_df.withColumn(
    "event_name",
    trim(col("event_name"))
)

display(
    marathon_silver_df.select(
        "event_name"
    )
)

- The event_name column has no null or empty values.
- For Silver, I trimmed event_name to remove possible leading or trailing spaces. This column will later be used to create event_id, which is needed for dimensional modelling.

## event_distance_length
#### Silver Layer cleaning rules are:
- if event has unit km or mi then performance should be in h
- if event has unit h then performance should be in km

**So there's 2 types of events:**
- distance event: km / mi
- time event: h

**Keep valid for OBT:**
- km
- mi
- h
- k / K, but standardize to km

**Remove invalid:**
- Etappen / multi-stage events
- d / days, for example 6d, 7d, 10d, 18d
- None
- malformed values like 07:35, 6:40, 71.5, 45.1m

**Next task sequence after event_distance_length:**
1. Clean event_distance_length
2. Create event_type
3. Standardize event distance/unit
4. Validate athlete_performance against event_type

In [0]:
# Classify event_distance_length before cleaning
event_distance_type_df = (
    marathon_silver_df
    .withColumn(
        "event_distance_type",
        when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+(\.[0-9]+)?km$"), "distance_km")
        .when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+(\.[0-9]+)?mi$"), "distance_mi")
        .when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+h$"), "time_hours")
        .when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+(\.[0-9]+)?k$"), "distance_k_needs_standardization")
        .when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+d$"), "days_invalid")
        .when(lower(trim(col("event_distance_length"))).contains("etappen"), "multi_stage_invalid")
        .otherwise("invalid_or_unknown")
    )
    .groupBy("event_distance_type")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(event_distance_type_df)